# Saving metadata
Following Dublic Core Metadata Initiative ([DCMI](https://www.ietf.org/rfc/rfc2413.txt)). Basic metadata categories to be included:
- Title
- Creator - Author/creator of content
- Subject - keywords/phrases describing subject or resource content. Use of controlled vocabularies is preferred here.
- Description - May include an abstract or a simple description of the resource content
- Publisher - Entity (publisher, university, corporation) that made content available
- Contributor - Person/organisation that made a significant (but secondary) contribution to creation of resource
- Date - When was the resource created, should follow format YYYY-MM-DD
- Type - Category of resource (e.g., technical report, paper, etc.)
- Format - Data format and may include dimensions of resource (e.g., size, duration)
- Identifier - Unique and persistent identifier such as DOI
- Source - Information about a second resource from which the present resource is derived.
- Language
- Relation - Secondary resource linked to resource being described (e.g., paper) and its relationship
- Coverage - Spatio-temporal. Can use geographical names, coordinates. Time periods or dates in YYYY-MM-DD format
- Rights - A rights management statement, an identifier that links to a rights management statement (e.g., license)

More info: https://www.dublincore.org/specifications/dublin-core/dcmi-terms/
Other potentially relevant standards:
- Biodiversity Information Standards (TDWG): https://www.tdwg.org/
- Ecological Metadata Language (EML): https://eml.ecoinformatics.org/
- Bioschemas: https://bioschemas.org/profiles/Dataset/1.1-DRAFT

In [1]:
import xarray as xr
import numpy as np
# import matplotlib.pyplot as plt
from glob import glob
import os

In [2]:
fao_numb = 88

In [3]:
res = '025deg'

In [29]:
var = 'detritivores'
var

'detritivores'

In [30]:
var_name = 'detritivores'

In [6]:
base_dir = f'/g/data/vf71/fishmip_outputs/ISIMIP3a/fao_outputs/fao-{fao_numb}/fishing_runs'
base_dir

'/g/data/vf71/fishmip_outputs/ISIMIP3a/fao_outputs/fao-88/fishing_runs'

In [32]:
ds = xr.open_mfdataset(
    glob(os.path.join(base_dir, res, f'{var}*[1,2][9,0][0-9][0-9]*.nc')),
    parallel = True)[var_name].chunk({'time': 12})
ds

<xarray.DataArray 'detritivores' (time: 1332, size_class: 181, lat: 74, lon: 560)> Size: 80GB
dask.array<rechunk-merge, shape=(1332, 181, 74, 560), dtype=float64, chunksize=(12, 181, 74, 560), chunktype=numpy.ndarray>
Coordinates:
  * size_class  (size_class) float64 1kB -12.0 -11.9 -11.8 -11.7 ... 5.8 5.9 6.0
  * lat         (lat) float64 592B -60.12 -60.38 -60.62 ... -77.88 -78.12 -78.38
  * lon         (lon) float64 4kB 150.1 150.4 150.6 150.9 ... 289.4 289.6 289.9
  * time        (time) datetime64[ns] 11kB 1900-01-01 1900-02-01 ... 2010-12-01

In [33]:
ds = ds.sel(time = slice('1961', None)).chunk({'time': 30})#120 or 30
ds

<xarray.DataArray 'detritivores' (time: 600, size_class: 181, lat: 74, lon: 560)> Size: 36GB
dask.array<rechunk-merge, shape=(600, 181, 74, 560), dtype=float64, chunksize=(30, 181, 74, 560), chunktype=numpy.ndarray>
Coordinates:
  * size_class  (size_class) float64 1kB -12.0 -11.9 -11.8 -11.7 ... 5.8 5.9 6.0
  * lat         (lat) float64 592B -60.12 -60.38 -60.62 ... -77.88 -78.12 -78.38
  * lon         (lon) float64 4kB 150.1 150.4 150.6 150.9 ... 289.4 289.6 289.9
  * time        (time) datetime64[ns] 5kB 1961-01-01 1961-02-01 ... 2010-12-01

In [34]:
out_fn = os.path.join(base_dir, res, f'{var}_{res}_fao-{fao_numb}_1961-2010.zarr')
out_fn

'/g/data/vf71/fishmip_outputs/ISIMIP3a/fao_outputs/fao-88/fishing_runs/025deg/detritivores_025deg_fao-88_1961-2010.zarr'

In [35]:
ds.to_zarr(out_fn, consolidated = True, mode = 'w')